# Data Quality Checks (Round 2)


In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import sys
sys.path.append('../src')
from config import SQLALCHEMY_URL

engine = create_engine(SQLALCHEMY_URL)
print('connected')

In [ ]:
customers = pd.read_sql('SELECT * FROM dim_customers', engine)
products = pd.read_sql('SELECT * FROM dim_products', engine)
sellers = pd.read_sql('SELECT * FROM dim_sellers', engine)
orders = pd.read_sql('SELECT * FROM dim_orders', engine)
order_items = pd.read_sql('SELECT * FROM fact_order_items', engine)
payments = pd.read_sql('SELECT * FROM fact_payments', engine)
reviews = pd.read_sql('SELECT * FROM fact_reviews', engine)

print('customers:', customers.shape)
print('products:', products.shape)
print('sellers:', sellers.shape)
print('orders:', orders.shape)
print('order_items:', order_items.shape)
print('payments:', payments.shape)
print('reviews:', reviews.shape)

In [ ]:
print('--- orders nulls ---')
print(orders.isnull().sum())

In [ ]:
# Does every 'delivered' order actually have a delivery date?
delivered = orders[orders['order_status'] == 'delivered']
missing_delivery_date = delivered['order_delivered_customer_date'].isnull().sum()
print(f"Orders marked 'delivered' but missing a delivery date: {missing_delivery_date}")

In [ ]:
print(order_items['price'].describe())
print()
print('Items priced at 0 or less:', (order_items['price'] <= 0).sum())
print('Top 5 highest priced items:')
print(order_items.sort_values('price', ascending=False).head(5)[['product_id', 'price']])

In [ ]:
dupe_check = customers.groupby('customer_unique_id')['customer_id'].nunique()
print('Customers with more than one customer_id:', (dupe_check > 1).sum())

In [ ]:
orphan_items = ~order_items['order_id'].isin(orders['order_id'])
print('Order items with no matching order:', orphan_items.sum())

## Summary
- Row counts matched expectations: yes/no
- Null patterns made sense: yes/no, notes
- Outliers found: none / list them
- Duplicate customers found: X
- Orphan records found: X